# MyMentorLLM — analysis code

Recomputes every table, figure and in-text number of the paper from the public release, and asserts each
against the published value. 

In [ ]:
from pathlib import Path

import pandas as pd
from IPython.display import display

import analysis as A
import emotions as E
import figures as F

DATA_DIR = "hf://datasets/RodolfoRizzi/MyMentorLLM-dataset/data"  # or a local folder, see README
OUT_DIR = Path("results")
OUT_DIR.mkdir(exist_ok=True)

df = A.load_sessions(DATA_DIR)
human_items, human_hist = A.load_human_ctrs()
assert len(df) == 2100 and df.groupby(["cond", "patient_case"]).size().eq(100).all()


def save(fig, name):
    fig.savefig(OUT_DIR / name, dpi=300, bbox_inches="tight")


def check_text(values):
    for name, (got, expected) in values.items():
        assert round(got, 1) == expected, f"{name}: {got:.2f} vs {expected} in the paper"
    print(f"OK: {len(values)} in-text values match the paper.")

## Therapeutic competence (CTRS) — Table 1, Fig. 3

In [ ]:
table1 = A.ctrs_table(df, human_items)
display(table1.round(2))
A.check(table1, pd.read_csv(A.REFERENCE / "paper_table1.csv", index_col="item"), 2)
table1.round(2).to_csv(OUT_DIR / "Table1_ctrs_means.csv")

In [ ]:
save(F.ctrs_distributions(df, human_items, human_hist), "Fig3a_ctrs_score_distributions_vs_human.pdf")
save(F.ctrs_boxplots(df, human_hist), "Fig3bc_ctrs_total_by_disorder_and_condition.pdf")

## Diagnosis before and after the mentor's feedback — Fig. 4

Exact scoring: an answer is correct only if it names the true disorder alone; hedged (`Multiple`) answers and
refusals count as wrong. `c` is the normalised change of Marx & Cummings (2007).

In [ ]:
paper = pd.read_csv(A.REFERENCE / "paper_fig4a.csv", index_col="cond")
fig4a = A.diagnosis_table(df).loc[paper.index]
display(fig4a.round(2))
A.check(fig4a[["A_I", "A_F", "A_S"]], paper[["A_I", "A_F", "A_S"]], 1)
A.check(fig4a[["c"]], paper[["c"]], 2)
fig4a.round(2).to_csv(OUT_DIR / "Fig4a_diagnosis_and_symptom_accuracy.csv")

In [ ]:
by_dx = df.groupby("patient_case")[["trainee_q1_correct", "trainee_q3_correct"]].mean() * 100
initial = pd.crosstab(df["patient_case"], df["trainee_q1_dx"], normalize="index") * 100
final = pd.crosstab(df["patient_case"], df["trainee_q3_dx"], normalize="index") * 100
sycophancy = A.feedback_outcomes(df)["sycophancy"] * 100
check_text({
    "GAD initial": (by_dx.loc["GAD", "trainee_q1_correct"], 97.7),
    "MDD initial": (by_dx.loc["MDD", "trainee_q1_correct"], 91.0),
    "BPD initial": (by_dx.loc["BPD", "trainee_q1_correct"], 65.3),
    "BPD -> MDD initial": (initial.loc["BPD", "MDD"], 15.4),
    "BPD -> Multiple initial": (initial.loc["BPD", "Multiple"], 14.7),
    "BPD final": (by_dx.loc["BPD", "trainee_q3_correct"], 82.6),
    "GAD final": (by_dx.loc["GAD", "trainee_q3_correct"], 97.9),
    "MDD final": (by_dx.loc["MDD", "trainee_q3_correct"], 82.6),
    "MDD -> GAD initial": (initial.loc["MDD", "GAD"], 2.0),
    "MDD -> GAD final": (final.loc["MDD", "GAD"], 11.3),
})
assert (fig4a["c"] > 0).sum() == 5
assert sycophancy.filter(like="E2B").round().between(12, 16).all()
display(sycophancy.round(1).rename("sycophancy (%)"))

In [ ]:
save(F.feedback_benefit_harm(A.feedback_outcomes(df)), "Fig4b_feedback_beneficial_vs_harmful.pdf")
save(F.confusion_matrices(df), "Fig4cd_diagnosis_confusion_before_after_feedback.pdf")
save(F.diagnosis_flows(df), "Fig4ef_diagnosis_transitions_after_feedback.pdf")

## Symptom identification — Fig. 5

A session scores the share of its five named symptoms that fall in the true disorder's block of the 35-item
list (1–9 BPD, 10–26 MDD, 27–35 GAD); naming any number other than five scores 0. `A_S` corrects each disorder
for chance (9/35, 17/35, 9/35) and averages the three.

In [ ]:
save(F.symptom_by_disorder(df), "Fig5a_symptom_accuracy_by_condition_and_disorder.pdf")
save(F.symptom_by_dx_correctness(df), "Fig5b_symptom_accuracy_by_final_diagnosis.pdf")

## Emotional signatures (EmoAtlas) — Fig. 2, Supplementary Table 1

For each condition × disorder × role, the spoken turns of the 100 sessions are pooled and EmoAtlas scores the
eight Plutchik emotions as z-scores against its Monte-Carlo null (300 samples, seed 42); |z| > 1.96 is
significant. The 42 texts take about 1.5 hours; the z-scores are cached in `results/` and reused on later
runs (delete the file to recompute). The Human column is the pooled HOPE corpus: its z-scores are shipped, and are recomputed when
`HOPE_DIR` points to the HOPE session files (see README).

In [ ]:
z_file = OUT_DIR / "emoatlas_zscores_raw.csv"
if z_file.exists():
    z = pd.read_csv(z_file)
else:
    z = E.zscores(E.pooled_texts(pd.read_parquet(f"{DATA_DIR}/therapy_turns.parquet")))
    z.to_csv(z_file, index=False)

HOPE_DIR = None  # folder with the 212 HOPE session CSVs (from the HOPE authors) to recompute the Human column
if HOPE_DIR:
    human = E.zscores(E.hope_texts(HOPE_DIR), language="english").set_index("role")[E.EMOTIONS]
else:
    human = pd.read_csv(A.REFERENCE / "hope_zscores.csv", index_col="role")
supp = E.supplementary_table(z, human)
supp.round(2).to_csv(OUT_DIR / "SuppTable1_emoatlas_zscores.csv")
A.check(supp, pd.read_csv(A.REFERENCE / "paper_supp_table1.csv", index_col=["disorder", "role", "emotion"]), 2)

In [ ]:
significant = supp.drop(columns="Human").abs().gt(E.SIGNIFICANCE).sum(axis=1)
check_text({
    "MDD patient sadness": (significant["MDD", "patient", "sadness"], 7),
    "GAD patient fear": (significant["GAD", "patient", "fear"], 5),
    "GAD patient anticipation": (significant["GAD", "patient", "anticipation"], 7),
    "BPD patient fear": (significant["BPD", "patient", "fear"], 7),
    "BPD patient sadness": (significant["BPD", "patient", "sadness"], 7),
    "BPD patient anger": (significant["BPD", "patient", "anger"], 5),
})
display(significant.xs("therapist", level="role").unstack().rename_axis(columns="therapist: significant conditions (of 7)"))

In [ ]:
E.draw_fig2(z, human, OUT_DIR / "Fig2_emotional_flowers")